# 04. Insight Bisnis dari Prediksi Model

Notebook ini menghitung insight bisnis dari **prediksi model terpilih** dan mengagregasinya per coffee shop, sesuai `PROJECT_CONTEXT.md §9`. Ini adalah tahap setelah modelling dan menjadi sumber utama `RESULTS.md` untuk penyusunan paper.

Model terpilih (keputusan dari notebook 03):

| Task | Model terpilih | Alasan |
|---|---|---|
| A - deteksi aspek | SVM (`OneVsRestClassifier(LinearSVC)`) | macro-F1 test tertinggi (0.8775) |
| B - sentimen per aspek | Random Forest per-aspek (6 model) | pooled macro-F1 test tertinggi (0.7013) dan recall `negative` tertinggi (0.5283) |

Keluaran notebook ini:

- prediksi model terpilih untuk **seluruh** data (train+validation+test, bukan hanya test) di `data/processed/insights/`;
- agregasi per coffee shop (mention aspek, distribusi sentimen, kekuatan, keluhan, prioritas perbaikan) di `reports/tables/insights/`;
- visualisasi di `reports/figures/insights/`;
- `RESULTS.md` di root repo - dokumen hasil lengkap untuk penulisan paper;
- `insights_manifest.json` - ringkasan reproducible dari tahap ini.

**Keterbatasan yang melekat** (lihat `PROJECT_CONTEXT.md §10`): label adalah pseudo-label berbasis aturan leksikal, bukan human gold; insight adalah agregasi **prediksi model**, bukan ground truth; rating Google hanya dipakai sebagai sanity check eksternal yang dilaporkan terpisah.

In [ ]:
from __future__ import annotations

import hashlib
import json
from datetime import datetime
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 25)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300})

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise RuntimeError(f"Jalankan notebook dari root repo, cwd sekarang: {cwd}")
AUTHOR = "Robby"
TEXT_VARIANT = "text_clean"
SPLITS = ["train", "validation", "test"]
ASPECTS = [
    "product_quality",
    "service",
    "price_value",
    "ambience",
    "facilities",
    "cleanliness",
]
ASPECT_LABEL_COLUMNS = [f"aspect_{aspect}" for aspect in ASPECTS]
SENTIMENTS = ["negative", "neutral", "positive"]
SENTIMENT_CODE = {"negative": -1, "neutral": 0, "positive": 1}

# Ambang minimum support: sebuah (shop, aspek) hanya masuk analisis kekuatan/keluhan
# bila memiliki setidaknya MIN_SUPPORT mention, agar satu review tidak mendominasi keputusan.
MIN_SUPPORT = 3

# Jalur input
TASK_A_INPUT = PROJECT_ROOT / "data/processed/modeling/splits/aspect_detection_with_split.csv"
TASK_B_INPUT = PROJECT_ROOT / "data/processed/modeling/splits/aspect_sentiment_with_split.csv"
SPLIT_MANIFEST = PROJECT_ROOT / "data/processed/modeling/splits/split_manifest.json"
SHOP_LOOKUP = PROJECT_ROOT / "data/processed/labeling/shop_id_lookup.csv"
RATING_RAW = PROJECT_ROOT / "data/raw/google_maps_reviews_jatinangor_raw.csv"

# Artefak model (vectorizer TF-IDF dari MNB, model terpilih dari SVM/RF)
MNB_DIR = PROJECT_ROOT / "data/processed/modeling/mnb"
SVM_DIR = PROJECT_ROOT / "data/processed/modeling/svm"
RF_DIR = PROJECT_ROOT / "data/processed/modeling/rf"

# Keluaran
INSIGHTS_DATA_DIR = PROJECT_ROOT / "data/processed/insights"
INSIGHTS_TABLES_DIR = PROJECT_ROOT / "reports/tables/insights"
INSIGHTS_FIGURES_DIR = PROJECT_ROOT / "reports/figures/insights"
for directory in (INSIGHTS_DATA_DIR, INSIGHTS_TABLES_DIR, INSIGHTS_FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RESULTS_MD = PROJECT_ROOT / "RESULTS.md"
print("Setup selesai:", PROJECT_ROOT)


## 1. Memuat data, lookup shop, dan rating mentah

Notebook hanya **membaca** data; tidak membuat ulang split atau melatih ulang model. Verifikasi hash input sama dengan notebook 03: hash file split harus cocok dengan `split_manifest.json` (line ending dinormalisasi bila perlu).

Rating mentah dipakai **hanya** sebagai sanity check eksternal pada sel berikutnya, bukan sebagai target.

In [ ]:
task_a = pd.read_csv(TASK_A_INPUT, encoding="utf-8-sig", keep_default_na=False)
task_b = pd.read_csv(TASK_B_INPUT, encoding="utf-8-sig", keep_default_na=False)
shop_lookup = pd.read_csv(SHOP_LOOKUP, encoding="utf-8-sig", keep_default_na=False)
rating_raw = pd.read_csv(RATING_RAW, keep_default_na=False)

required_a = {"review_id", "coffee_shop_id", "duplicate_group_id", "split", TEXT_VARIANT,
              *ASPECT_LABEL_COLUMNS}
required_b = {"annotation_id", "review_id", "coffee_shop_id", "aspect_category",
              "sentiment", "split", TEXT_VARIANT}
missing_a = sorted(required_a - set(task_a.columns))
missing_b = sorted(required_b - set(task_b.columns))
if missing_a or missing_b:
    raise ValueError(f"Kolom hilang. Task A: {missing_a}; Task B: {missing_b}")

assert set(task_a["split"]) == set(SPLITS), "Nilai split Task A tidak valid"
assert set(task_b["split"]) == set(SPLITS), "Nilai split Task B tidak valid"
assert set(task_b["aspect_category"]) == set(ASPECTS), "aspect_category tidak sesuai"
assert set(task_b["sentiment"]) == set(SENTIMENTS), "Label sentimen tidak sesuai"
assert task_a["review_id"].is_unique and task_b["annotation_id"].is_unique

# Tidak ada group duplikat yang terbelah antar split.
for name, frame in (("Task A", task_a), ("Task B", task_b)):
    splits_per_group = frame.groupby("duplicate_group_id")["split"].nunique()
    assert (splits_per_group <= 1).all(), f"Group terbelah di {name}"


def sha256_file(path: Path, normalize_crlf: bool = False) -> str:
    data = Path(path).read_bytes()
    if normalize_crlf:
        data = data.replace(b"\r\n", b"\n").replace(b"\n", b"\r\n")
    return hashlib.sha256(data).hexdigest()


split_manifest = json.loads(SPLIT_MANIFEST.read_text(encoding="utf-8"))
expected_hashes = {
    key.replace("\\", "/"): value["sha256"]
    for key, value in split_manifest["outputs"].items()
}
input_hashes = {}
for path in (TASK_A_INPUT, TASK_B_INPUT):
    relative = path.relative_to(PROJECT_ROOT).as_posix()
    raw_hash = sha256_file(path)
    crlf_hash = sha256_file(path, normalize_crlf=True)
    expected = expected_hashes[relative]
    if raw_hash == expected:
        status = "match"
    elif crlf_hash == expected:
        status = "match_after_crlf_normalization"
    else:
        status = "MISMATCH"
    input_hashes[relative] = {"sha256_raw": raw_hash,
                              "sha256_crlf_normalized": crlf_hash,
                              "expected_in_split_manifest": expected,
                              "status": status}
assert all(item["status"] != "MISMATCH" for item in input_hashes.values()),     "Hash file split tidak cocok dengan split_manifest.json"

shop_name = shop_lookup.set_index("coffee_shop_id")["source_place_name"].to_dict()
assert set(task_a["coffee_shop_id"]) <= set(shop_name), "Ada shop Task A tanpa nama"
assert set(task_b["coffee_shop_id"]) <= set(shop_name), "Ada shop Task B tanpa nama"
assert task_a["review_id"].isin(rating_raw["review_id"]).all(),     "Tidak semua review_id Task A ada di rating mentah"

n_reviews = len(task_a)
n_annotations = len(task_b)
n_shops = len(shop_lookup)
print(f"Task A: {task_a.shape} | Task B: {task_b.shape}")
print(f"Shop terdaftar: {n_shops} | Rating mentah: {len(rating_raw)} baris")
print("Hash input:", {Path(k).name: v["status"] for k, v in input_hashes.items()})


## 2. Model terpilih

Muat vectorizer TF-IDF (diwarisi dari MNB, tidak di-fit ulang) dan model terpilih:
Task A = SVM, Task B = 6 Random Forest per-aspek. Tabel perbandingan tiga algoritma
dibaca langsung dari ketiga manifest modelling.

In [ ]:
tfidf_a = joblib.load(MNB_DIR / "task_a" / "tfidf_text_clean.joblib")
model_a = joblib.load(SVM_DIR / "task_a" / "model_svm.joblib")
tfidf_b = joblib.load(MNB_DIR / "task_b" / "tfidf_text_clean_per_aspect.joblib")
models_b = {
    aspect: joblib.load(RF_DIR / "task_b" / f"model_rf_{aspect}.joblib")
    for aspect in ASPECTS
}

mnb_manifest = json.loads((MNB_DIR / "mnb_manifest.json").read_text(encoding="utf-8"))
svm_manifest = json.loads((SVM_DIR / "svm_manifest.json").read_text(encoding="utf-8"))
rf_manifest = json.loads((RF_DIR / "rf_manifest.json").read_text(encoding="utf-8"))
manifests = {"MNB": mnb_manifest, "SVM": svm_manifest, "RF": rf_manifest}

comparison = pd.DataFrame(
    {
        "model": list(manifests),
        "task_a_macro_f1": [m["task_a"]["macro_f1"] for m in manifests.values()],
        "task_a_micro_f1": [m["task_a"]["micro_f1"] for m in manifests.values()],
        "task_b_per_aspect_pooled_macro_f1": [
            m["task_b"]["per_aspect_models"]["pooled_macro_f1"] for m in manifests.values()
        ],
        "task_b_per_aspect_recall_negative": [
            m["task_b"]["per_aspect_models"]["pooled_recall_negative"] for m in manifests.values()
        ],
        "task_b_combined_pooled_macro_f1": [
            m["task_b"]["combined_model"]["pooled_macro_f1"] for m in manifests.values()
        ],
    }
)
display(comparison.round(4))

print("Model terpilih:")
print("  Task A:", svm_manifest["model_config"]["task_a"])
print("  Task B:", rf_manifest["model_config"]["task_b_per_aspect"])


### 2.1 Inferensi Task A (deteksi aspek) pada seluruh review

Prediksi SVM untuk semua split digabung menjadi satu tabel: satu baris = satu review,
kolom `pred_*` = aspek yang terdeteksi model.

In [ ]:
A = {split: task_a[task_a["split"] == split].reset_index(drop=True) for split in SPLITS}

pred_a_parts = []
for split in SPLITS:
    X = tfidf_a.transform(A[split][TEXT_VARIANT])
    pred = model_a.predict(X)
    frame = A[split][["review_id", "coffee_shop_id"]].copy()
    for i, aspect in enumerate(ASPECTS):
        frame[f"pred_{aspect}"] = pred[:, i]
    frame["split"] = split
    pred_a_parts.append(frame)
pred_a_all = pd.concat(pred_a_parts, ignore_index=True)
pred_a_all["n_aspects_pred"] = pred_a_all[[f"pred_{a}" for a in ASPECTS]].sum(axis=1)
pred_a_all.to_csv(INSIGHTS_DATA_DIR / "task_a_predictions_all.csv", index=False)

print("Prediksi Task A:", pred_a_all.shape)
display(pred_a_all.head())
print("Review tanpa aspek terdeteksi:",
      int((pred_a_all["n_aspects_pred"] == 0).sum()),
      f"({(pred_a_all['n_aspects_pred'] == 0).mean():.2%})")


### 2.2 Inferensi Task B (sentimen per aspek) pada seluruh anotasi

Prediksi 6 model RF untuk semua split digabung: satu baris = satu pasangan
review-aspek. `sentiment_true` (pseudo-label) disimpan hanya untuk referensi;
seluruh insight memakai `sentiment_pred`.

Catatan cakupan: inferensi dilakukan pada pasangan review-aspek yang sudah ada di
dataset anotasi. Simulasi pipeline end-to-end (Task A dulu, lalu Task B pada aspek
yang baru terdeteksi) berada di luar cakupan tahap ini dan dicatat sebagai future work.

In [ ]:
B = {
    aspect: {
        split: task_b[(task_b["aspect_category"] == aspect) & (task_b["split"] == split)]
        .reset_index(drop=True)
        for split in SPLITS
    }
    for aspect in ASPECTS
}

pred_b_parts = []
for aspect in ASPECTS:
    for split in SPLITS:
        rows = B[aspect][split]
        X = tfidf_b[aspect].transform(rows[TEXT_VARIANT])
        pred = models_b[aspect].predict(X)
        pred_b_parts.append(
            pd.DataFrame(
                {
                    "annotation_id": rows["annotation_id"],
                    "review_id": rows["review_id"],
                    "coffee_shop_id": rows["coffee_shop_id"],
                    "aspect_category": aspect,
                    "sentiment_true": rows["sentiment"],
                    "sentiment_pred": pred,
                    "split": split,
                }
            )
        )
pred_b_all = pd.concat(pred_b_parts, ignore_index=True)
pred_b_all.to_csv(INSIGHTS_DATA_DIR / "task_b_predictions_all.csv", index=False)

print("Prediksi Task B:", pred_b_all.shape)
display(pred_b_all.head())
display(
    pred_b_all["sentiment_pred"].value_counts().reindex(SENTIMENTS).rename("n")
    .to_frame().assign(share=lambda f: f["n"] / f["n"].sum()).round(4)
)


### 2.3 Sanity check eksternal: korelasi rating vs sentimen prediksi

Rating Google **bukan** target dan tidak dipakai dalam pelatihan. Di sini rating hanya
dipakai sebagai sinyal eksternal: bila prediksi sentimen masuk akal, korelasi rank
Spearman antara rating (1-5) dan sentimen prediksi (negative=-1, neutral=0, positive=+1)
diharapkan positif.

In [ ]:
rating_map = rating_raw.set_index("review_id")["rating"].to_dict()
check = pred_b_all.copy()
check["rating"] = check["review_id"].map(rating_map)
assert check["rating"].notna().all(), "Ada review tanpa rating"
check["sentiment_code"] = check["sentiment_pred"].map(SENTIMENT_CODE)

rating_rows = []
for aspect in ASPECTS:
    sub = check[check["aspect_category"] == aspect]
    rho, p_value = spearmanr(sub["rating"], sub["sentiment_code"])
    rating_rows.append({
        "aspect_category": aspect,
        "n": int(len(sub)),
        "spearman_rho": float(rho),
        "p_value": float(p_value),
    })
rho, p_value = spearmanr(check["rating"], check["sentiment_code"])
rating_rows.append({
    "aspect_category": "ALL",
    "n": int(len(check)),
    "spearman_rho": float(rho),
    "p_value": float(p_value),
})
rating_check = pd.DataFrame(rating_rows)
rating_check.to_csv(INSIGHTS_TABLES_DIR / "rating_sanity_check.csv", index=False)
display(rating_check.round(4))
print("Interpretasi: rho positif = arah prediksi sentimen sejalan dengan rating Google.")


## 3. Agregasi per coffee shop

- **Mention aspek** dihitung dari prediksi Task A (SVM): berapa review tiap shop menyebut tiap aspek.
- **Distribusi sentimen** dihitung dari prediksi Task B (RF): jumlah negative/neutral/positive per (shop, aspek).
- **Skor sentimen** = (positive - negative) / total, bernilai -1 sampai +1.
- `MIN_SUPPORT = 3`: pasangan (shop, aspek) di bawah ambang ini tidak ikut analisis kekuatan/keluhan/prioritas.

In [ ]:
# 3.1 Mention aspek per shop (prediksi Task A)
mention_cols = [f"mention_{aspect}" for aspect in ASPECTS]
mentions = pred_a_all.groupby("coffee_shop_id").agg(
    n_reviews=("review_id", "nunique"),
    **{col: (f"pred_{aspect}", "sum") for col, aspect in zip(mention_cols, ASPECTS)},
).reset_index()
mentions["n_mentions_total"] = mentions[mention_cols].sum(axis=1)
mentions["shop_name"] = mentions["coffee_shop_id"].map(shop_name)

# 3.2 Distribusi sentimen per (shop, aspek) dari prediksi Task B
counts = (
    pred_b_all.groupby(["coffee_shop_id", "aspect_category", "sentiment_pred"])
    .size()
    .reset_index(name="n")
)
dist = counts.pivot_table(
    index=["coffee_shop_id", "aspect_category"],
    columns="sentiment_pred",
    values="n",
    fill_value=0,
).reset_index()
dist.columns = ["coffee_shop_id", "aspect_category",
                "n_negative", "n_neutral", "n_positive"]
dist["n_total"] = dist[["n_negative", "n_neutral", "n_positive"]].sum(axis=1)
dist["share_negative"] = dist["n_negative"] / dist["n_total"]
dist["share_neutral"] = dist["n_neutral"] / dist["n_total"]
dist["share_positive"] = dist["n_positive"] / dist["n_total"]
dist["score"] = (dist["n_positive"] - dist["n_negative"]) / dist["n_total"]
dist["shop_name"] = dist["coffee_shop_id"].map(shop_name)

mentions.to_csv(INSIGHTS_TABLES_DIR / "shop_aspect_mentions.csv", index=False)
dist.to_csv(INSIGHTS_TABLES_DIR / "shop_aspect_sentiment.csv", index=False)

print("Mentions:", mentions.shape, "| Distribusi shop-aspek:", dist.shape)
display(mentions.sort_values("n_reviews", ascending=False).head(5))
display(dist[dist["n_total"] >= MIN_SUPPORT].sort_values("score").head(5))


## 4. Kekuatan, keluhan, dan prioritas perbaikan

Aturan (deterministik):

- **Kekuatan utama** per shop: aspek dengan `share_positive` tertinggi (support >= `MIN_SUPPORT`); seri dipecah oleh support terbesar.
- **Keluhan utama** per shop: aspek dengan `n_negative` >= 2; bila tidak ada, turun ke `n_negative` >= 1; bila tetap tidak ada, shop dianggap tanpa keluhan.
- **Prioritas perbaikan**: semua (shop, aspek) dengan `n_negative` >= 1 dan support >= `MIN_SUPPORT`, diurutkan `n_negative` lalu `share_negative` menurun.

In [ ]:
qualified = dist[dist["n_total"] >= MIN_SUPPORT].copy()


def pick_strength(sub: pd.DataFrame) -> pd.Series:
    return sub.sort_values(
        ["share_positive", "n_total"], ascending=[False, False]
    ).iloc[0]


def pick_weakness(sub: pd.DataFrame):
    cand = sub[sub["n_negative"] >= 2]
    if cand.empty:
        cand = sub[sub["n_negative"] >= 1]
    if cand.empty:
        return None
    return cand.sort_values(
        ["n_negative", "share_negative"], ascending=[False, False]
    ).iloc[0]


sw_rows = []
for shop, sub in qualified.groupby("coffee_shop_id"):
    s = pick_strength(sub)
    w = pick_weakness(sub)
    sw_rows.append({
        "coffee_shop_id": shop,
        "shop_name": shop_name.get(shop, shop),
        "strength_aspect": s["aspect_category"],
        "strength_share_positive": round(float(s["share_positive"]), 4),
        "strength_n": int(s["n_total"]),
        "weakness_aspect": w["aspect_category"] if w is not None else "",
        "weakness_n_negative": int(w["n_negative"]) if w is not None else 0,
        "weakness_share_negative": round(float(w["share_negative"]), 4) if w is not None else 0.0,
    })
sw = pd.DataFrame(sw_rows).sort_values("coffee_shop_id").reset_index(drop=True)

priorities = qualified[qualified["n_negative"] >= 1].copy()
priorities = priorities.sort_values(
    ["n_negative", "share_negative", "n_total"], ascending=[False, False, False]
).reset_index(drop=True)
priorities = priorities[
    ["coffee_shop_id", "shop_name", "aspect_category", "n_negative",
     "n_total", "share_negative", "score"]
]

sw.to_csv(INSIGHTS_TABLES_DIR / "shop_strengths_weaknesses.csv", index=False)
priorities.to_csv(INSIGHTS_TABLES_DIR / "improvement_priorities.csv", index=False)

print(f"Kekuatan/keluhan per shop (support >= {MIN_SUPPORT}): {len(sw)} shop")
display(sw.head(10))
print("Top 10 prioritas perbaikan:")
display(priorities.head(10))


## 5. Visualisasi

In [ ]:
# 5.1 Heatmap skor sentimen per shop x aspek
pivot = dist.pivot_table(index="coffee_shop_id", columns="aspect_category", values="score")
pivot = pivot.reindex(columns=ASPECTS)
pivot["_mean"] = pivot.mean(axis=1)
pivot = pivot.sort_values("_mean", ascending=False).drop(columns="_mean")
labels = [shop_name.get(shop, shop) for shop in pivot.index]

fig, ax = plt.subplots(figsize=(9, 14))
sns.heatmap(
    pivot, ax=ax, cmap="RdYlGn", center=0, vmin=-1, vmax=1,
    yticklabels=labels, cbar_kws={"label": "Skor sentimen prediksi"},
)
ax.set_title("Skor sentimen per coffee shop (prediksi model terpilih)")
ax.set_xlabel("Aspek")
ax.set_ylabel("Coffee shop")
ax.tick_params(axis="y", labelsize=7)
fig.tight_layout()
fig.savefig(INSIGHTS_FIGURES_DIR / "shop_sentiment_heatmap.png", dpi=300)
plt.show()

# 5.2 Top 10 keluhan
top10 = priorities.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(
    top10["shop_name"] + " | " + top10["aspect_category"],
    top10["n_negative"], color="#c0392b",
)
ax.set_xlabel(f"Jumlah prediksi negative (support >= {MIN_SUPPORT})")
ax.set_title("Top 10 keluhan berdasarkan prediksi sentimen")
fig.tight_layout()
fig.savefig(INSIGHTS_FIGURES_DIR / "top_10_complaints.png", dpi=300)
plt.show()

# 5.3 Distribusi sentimen per aspek (global)
global_counts = (
    pred_b_all.groupby(["aspect_category", "sentiment_pred"]).size()
    .unstack(fill_value=0)
    .reindex(index=ASPECTS, columns=SENTIMENTS)
)
global_share = global_counts.div(global_counts.sum(axis=1), axis=0)
colors = {"negative": "#c0392b", "neutral": "#7f8c8d", "positive": "#27ae60"}
fig, ax = plt.subplots(figsize=(8, 4.5))
bottom = np.zeros(len(ASPECTS))
for sent in SENTIMENTS:
    ax.bar(ASPECTS, global_share[sent], bottom=bottom, label=sent, color=colors[sent])
    bottom += global_share[sent].to_numpy()
ax.set_ylabel("Proporsi prediksi")
ax.set_title("Distribusi sentimen prediksi per aspek (semua coffee shop)")
ax.legend(title="Sentimen")
fig.tight_layout()
fig.savefig(INSIGHTS_FIGURES_DIR / "aspect_sentiment_distribution.png", dpi=300)
plt.show()


## 6. Manifest insight

Ringkasan reproducible: model yang dipakai, ambang support, hash input, dan daftar artefak.

In [ ]:
output_files = [
    "data/processed/insights/task_a_predictions_all.csv",
    "data/processed/insights/task_b_predictions_all.csv",
    "data/processed/insights/insights_manifest.json",
    "reports/tables/insights/shop_aspect_mentions.csv",
    "reports/tables/insights/shop_aspect_sentiment.csv",
    "reports/tables/insights/shop_strengths_weaknesses.csv",
    "reports/tables/insights/improvement_priorities.csv",
    "reports/tables/insights/rating_sanity_check.csv",
    "reports/figures/insights/shop_sentiment_heatmap.png",
    "reports/figures/insights/top_10_complaints.png",
    "reports/figures/insights/aspect_sentiment_distribution.png",
    "RESULTS.md",
]
now = datetime.now().astimezone()
insight_manifest = {
    "date": now.strftime("%Y-%m-%d"),
    "generated_at": now.isoformat(timespec="seconds"),
    "author": AUTHOR,
    "label_type": "pseudo-label (auto:lexical_preannotation_v1), bukan human gold",
    "source": "Prediksi model terpilih atas seluruh baris train+validation+test.",
    "selected_models": {
        "task_a": {
            "algorithm": svm_manifest["model_config"]["task_a"],
            "test_macro_f1": svm_manifest["task_a"]["macro_f1"],
            "test_micro_f1": svm_manifest["task_a"]["micro_f1"],
            "artifact": "data/processed/modeling/svm/task_a/model_svm.joblib",
        },
        "task_b": {
            "algorithm": rf_manifest["model_config"]["task_b_per_aspect"],
            "test_pooled_macro_f1": rf_manifest["task_b"]["per_aspect_models"]["pooled_macro_f1"],
            "test_pooled_recall_negative": rf_manifest["task_b"]["per_aspect_models"]["pooled_recall_negative"],
            "artifact": "data/processed/modeling/rf/task_b/",
        },
    },
    "min_support": MIN_SUPPORT,
    "inputs": {
        "splits": input_hashes,
        "rating_raw_rows": int(len(rating_raw)),
        "shop_lookup_rows": int(len(shop_lookup)),
    },
    "outputs": output_files,
}
manifest_path = INSIGHTS_DATA_DIR / "insights_manifest.json"
manifest_path.write_text(
    json.dumps(insight_manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Manifest ditulis:", manifest_path.relative_to(PROJECT_ROOT).as_posix())
display(pd.DataFrame([
    {"tanggal": insight_manifest["date"], "author": AUTHOR,
     "min_support": MIN_SUPPORT, "n_outputs": len(output_files)},
]))


## 7. Menyusun RESULTS.md

Dokumen hasil lengkap ditulis ke `RESULTS.md` di root repo: ringkasan eksekutif, metodologi,
hasil modeling + pemilihan model, insight bisnis, rekomendasi, batasan, dan lampiran profil
coffee shop. Dokumen ini menjadi bahan utama penulisan paper.

In [ ]:
def fmt(x, digits=4):
    return f"{x:.{digits}f}"


def md_table(df, columns=None, formatters=None):
    cols = list(columns or df.columns)
    head = "| " + " | ".join(cols) + " |"
    sep = "|" + "|".join(["---"] * len(cols)) + "|"
    lines = [head, sep]
    for _, row in df.iterrows():
        cells = []
        for c in cols:
            v = row[c]
            if formatters and c in formatters:
                v = formatters[c](v)
            elif isinstance(v, float):
                v = str(int(v)) if pd.notna(v) and v.is_integer() else fmt(v)
            cells.append(str(v))
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)


def round_floats(df):
    out = df.copy()
    for column in out.columns:
        if pd.api.types.is_float_dtype(out[column]):
            out[column] = out[column].round(4)
    return out


# --- Angka global ---
global_mentions = pred_a_all[[f"pred_{a}" for a in ASPECTS]].sum()
global_counts = (
    pred_b_all.groupby(["aspect_category", "sentiment_pred"]).size()
    .unstack(fill_value=0)
    .reindex(index=ASPECTS, columns=SENTIMENTS)
)
global_share = global_counts.div(global_counts.sum(axis=1), axis=0)
global_score = (global_counts["positive"] - global_counts["negative"]) / global_counts.sum(axis=1)

aspect_overview = pd.DataFrame({
    "aspect": ASPECTS,
    "mentions_task_a": [int(global_mentions[f"pred_{a}"]) for a in ASPECTS],
    "n_negative": [int(global_counts.loc[a, "negative"]) for a in ASPECTS],
    "n_neutral": [int(global_counts.loc[a, "neutral"]) for a in ASPECTS],
    "n_positive": [int(global_counts.loc[a, "positive"]) for a in ASPECTS],
})
aspect_overview["n_total"] = aspect_overview[["n_negative", "n_neutral", "n_positive"]].sum(axis=1)
aspect_overview["share_negative"] = aspect_overview["n_negative"] / aspect_overview["n_total"]
aspect_overview["score"] = global_score.to_numpy()

top_negative_aspects = (
    aspect_overview.sort_values(["n_negative", "share_negative"], ascending=[False, False])
    .head(3)
)
top_positive_aspects = (
    aspect_overview.assign(_sp=aspect_overview["n_positive"] / aspect_overview["n_total"])
    .sort_values("_sp", ascending=False)
    .head(3)
)

shop_score = dist.groupby("coffee_shop_id")["score"].mean().rename("mean_score").reset_index()
profile = (
    mentions.merge(shop_score, on="coffee_shop_id")
    .merge(sw[["coffee_shop_id", "strength_aspect", "weakness_aspect"]],
           on="coffee_shop_id", how="left")
    .sort_values("mean_score", ascending=False)
    .reset_index(drop=True)
)
profile_cols = ["shop_name", "n_reviews", "n_mentions_total", "mean_score",
                "strength_aspect", "weakness_aspect"]

# --- Susun dokumen ---
L = []
add = L.append
today = datetime.now().astimezone().strftime("%Y-%m-%d")

no_aspect = int((pred_a_all["n_aspects_pred"] == 0).sum())
no_aspect_share = no_aspect / len(pred_a_all)

add("# BrewScope Coffee Insights - Hasil Analisis")
add("")
add(f"> Dibuat otomatis oleh `notebooks/04_business_insights.ipynb` pada {today} oleh {AUTHOR}.")
add(">")
add("> **Status label:** pseudo-label berbasis aturan leksikal "
    "(`auto:lexical_preannotation_v1`), **bukan** human gold standard. "
    "Seluruh metrik dan insight mengacu pada prediksi model yang dilatih terhadap pseudo-label tersebut.")
add("")

add("## Ringkasan Eksekutif")
add("")
add(f"- Dataset: **{n_reviews} review** Google Maps dari **{n_shops} coffee shop** di Jatinangor; "
    f"Task B memiliki **{n_annotations} pasangan review-aspek**.")
add(f"- **Task A (deteksi aspek)**: model terbaik adalah **SVM** dengan macro-F1 test "
    f"{fmt(svm_manifest['task_a']['macro_f1'])} (micro-F1 {fmt(svm_manifest['task_a']['micro_f1'])}).")
add(f"- **Task B (sentimen per aspek)**: model terbaik adalah **Random Forest per-aspek** dengan pooled "
    f"macro-F1 test {fmt(rf_manifest['task_b']['per_aspect_models']['pooled_macro_f1'])} dan recall "
    f"`negative` {fmt(rf_manifest['task_b']['per_aspect_models']['pooled_recall_negative'])}.")
add(f"- Keluhan yang paling sering diprediksi: "
    f"**{', '.join(top_negative_aspects['aspect'].tolist())}**.")
add(f"- Aspek dengan sentimen terbaik: **{', '.join(top_positive_aspects['aspect'].tolist())}**.")
add(f"- Prioritas perbaikan disusun per (shop, aspek) dengan minimum support {MIN_SUPPORT} mention.")
add("")
add(f"Catatan cakupan: {no_aspect} dari {len(pred_a_all)} review ({no_aspect_share:.2%}) "
    "tidak mendapat aspek terdeteksi dari model Task A; review tersebut tetap masuk "
    "tabel lampiran `task_a_predictions_all.csv` tetapi tidak menyumbang sentimen aspek.")
add("")

add("## 1. Konteks Proyek")
add("")
add("BrewScope menganalisis ulasan Google Maps coffee shop di Jatinangor dengan pendekatan "
    "Aspect-Based Sentiment Analysis (ABSA) untuk menghasilkan prioritas perbaikan strategis "
    "per coffee shop. Pipeline: scraping -> preprocessing -> labeling -> splitting -> modelling -> "
    "**insight bisnis** (dokumen ini). Detail kontrak: `PROJECT_CONTEXT.md` dan `README.md`.")
add("")

add("## 2. Data dan Metodologi")
add("")
add("| Elemen | Deskripsi |")
add("|---|---|")
add(f"| Data Task A | {n_reviews} review, target 6 label biner aspek |")
add(f"| Data Task B | {n_annotations} pasangan review-aspek, 3 kelas sentimen |")
add("| Split | 70/15/15, dikelompokkan berdasar `duplicate_group_id` (0 kebocoran) |")
add("| Label | pseudo-label leksikal deterministik (`label_brewscope.py`) |")
add(f"| Fitur | TF-IDF `{TEXT_VARIANT}` (min_df=2, unigram+bigram, maks 10.000 fitur) |")
add("| Algoritma | MultinomialNB, LinearSVC (SVM), Random Forest |")
add(f"| Model terpilih | Task A: SVM; Task B: RF per-aspek (baseline utama) |")
add(f"| Ambang insight | minimum support {MIN_SUPPORT} mention per (shop, aspek) |")
add("")

add("## 3. Hasil Modelling dan Pemilihan Model")
add("")
add("### 3.1 Perbandingan tiga algoritma (test, pseudo-label)")
add("")
comparison_md = round_floats(comparison)
add(md_table(comparison_md))
add("")
add("### 3.2 Keputusan")
add("")
add("- **Task A**: SVM dipilih karena macro-F1 tertinggi sekaligus micro-F1 tertinggi.")
add("- **Task B**: RF per-aspek dipilih karena pooled macro-F1 dan recall `negative` tertinggi - "
    "paling penting untuk mendeteksi keluhan.")
add("")

add("## 4. Insight Bisnis")
add("")
add("### 4.1 Gambaran umum per aspek (prediksi model terpilih)")
add("")
add(md_table(round_floats(aspect_overview)))
add("")
add("### 4.2 Kekuatan utama per shop")
add("")
sw_show = sw[["shop_name", "strength_aspect", "strength_share_positive", "strength_n"]].head(10)
add(md_table(round_floats(sw_show)))
add("")
add("Tabel lengkap 40 shop: `reports/tables/insights/shop_strengths_weaknesses.csv`.")
add("")
add("### 4.3 Keluhan utama dan prioritas perbaikan (top 10)")
add("")
add(md_table(round_floats(priorities.head(10))))
add("")
add("### 4.4 Sanity check rating eksternal")
add("")
add("Rating Google tidak dipakai sebagai target; korelasi Spearman berikut hanya sanity check:")
add("")
add(md_table(round_floats(rating_check)))
add("")
add(f"Korelasi keseluruhan rho = {fmt(rating_check.iloc[-1]['spearman_rho'])}: arah prediksi "
    "sentimen model sejalan dengan rating pengguna.")
add("")

add("## 5. Rekomendasi Strategis")
add("")
for _, row in top_negative_aspects.iterrows():
    add(f"- Perhatikan **{row['aspect']}**: {int(row['n_negative'])} prediksi negative "
        f"({fmt(row['share_negative'] * 100, 1)}% dari mention aspek ini). "
        f"Prioritas tertinggi pada shop di tabel 4.3.")
add("")
add(f"- Pertahankan kekuatan pada **{', '.join(top_positive_aspects['aspect'].tolist())}** - "
    "aspek dengan proporsi positive tertinggi.")
add("- Gunakan tabel `improvement_priorities.csv` sebagai daftar kerja per shop; "
    "filter `n_negative >= 2` untuk sinyal yang lebih konservatif.")
add("")

add("## 6. Batasan")
add("")
add("- Label adalah pseudo-label berbasis aturan leksikal, bukan human gold.")
add("- Evaluasi mengukur kemampuan model mereproduksi pseudo-label.")
add("- Insight adalah agregasi prediksi model, bukan ground truth sentimen.")
add("- Distribusi sentimen sangat tidak seimbang ke arah positive.")
add("- Beberapa aspek memiliki support `negative` yang rendah; interpretasi keluhan "
    "dengan support kecil harus hati-hati.")
add("- Satu review non-Latin dikeluarkan dari modeling (tetap disimpan untuk audit).")
add("- Teks dapat mengandung code-switching Indonesia-Inggris.")
add("- Inferensi Task B dilakukan pada pasangan review-aspek yang sudah ada di dataset anotasi; "
    "pipeline end-to-end (Task A lalu Task B pada review baru) adalah future work.")
add("")

add("## 7. Artefak dan Cara Reproduksi")
add("")
add("Jalankan dari awal: `notebooks/03_modeling_all_models.ipynb` lalu "
    "`notebooks/04_business_insights.ipynb` (Python 3.11, environment `brewscope`).")
add("")
add("Artefak utama tahap insight:")
add("")
for f in output_files:
    add(f"- `{f}`")
add("")

add("## Lampiran: Profil Coffee Shop")
add("")
profile_show = profile[profile_cols]
add(md_table(round_floats(profile_show)))
add("")

RESULTS_MD.write_text("\n".join(L) + "\n", encoding="utf-8")
print("RESULTS.md ditulis:", RESULTS_MD)
print(f"Panjang: {len(chr(10).join(L))} karakter, {len(L)} baris")


In [ ]:
expected = [
    INSIGHTS_DATA_DIR / "task_a_predictions_all.csv",
    INSIGHTS_DATA_DIR / "task_b_predictions_all.csv",
    INSIGHTS_DATA_DIR / "insights_manifest.json",
    INSIGHTS_TABLES_DIR / "shop_aspect_mentions.csv",
    INSIGHTS_TABLES_DIR / "shop_aspect_sentiment.csv",
    INSIGHTS_TABLES_DIR / "shop_strengths_weaknesses.csv",
    INSIGHTS_TABLES_DIR / "improvement_priorities.csv",
    INSIGHTS_TABLES_DIR / "rating_sanity_check.csv",
    INSIGHTS_FIGURES_DIR / "shop_sentiment_heatmap.png",
    INSIGHTS_FIGURES_DIR / "top_10_complaints.png",
    INSIGHTS_FIGURES_DIR / "aspect_sentiment_distribution.png",
    RESULTS_MD,
]
missing = [p.relative_to(PROJECT_ROOT).as_posix() for p in expected if not p.is_file()]
assert not missing, f"Artefak hilang: {missing}"

# Konsistensi internal: setiap anotasi Task B punya review di Task A.
assert set(pred_b_all["review_id"]) <= set(pred_a_all["review_id"])

print("Semua pemeriksaan lolos.")
print("Ringkasan:")
print(f"  Review (Task A): {len(task_a)} | Anotasi (Task B): {len(task_b)} | Shop: {n_shops}")
print(f"  Task A terpilih: SVM | macro-F1 test {fmt(svm_manifest['task_a']['macro_f1'])}")
print(f"  Task B terpilih: RF per-aspek | pooled macro-F1 test "
      f"{fmt(rf_manifest['task_b']['per_aspect_models']['pooled_macro_f1'])}")
print(f"  Prioritas perbaikan: {len(priorities)} baris (support >= {MIN_SUPPORT})")
print("  RESULTS.md:", RESULTS_MD)
